In [1]:
!pip install "tensorboard~=2.19.0"


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf, tensorboard as tb
print(tf.__version__)
print(tb.__version__)


In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer gradio datasets pyarrow


In [ ]:
!pip install -U torchaudio torchvision


In [ ]:
!pip install -U torchcodec torch

In [ ]:
!pip install transformers==4.52.0 --quiet

Reason for being yanked: <none given>


In [ ]:
import torch, torchaudio, torchvision
print(torch.__version__)
print(torchaudio.__version__)
print(torchvision.__version__)


2.9.1+cu128
2.9.1+cu128
0.24.1+cu128


In [ ]:

import datasets
print(datasets.__version__)



4.4.1


In [ ]:
import transformers
print(transformers.__version__)


4.52.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
datasetdict = load_dataset("ysdede/commonvoice_17_tr_fixed")

In [ ]:
print(datasetdict)

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 26501
    })
    test: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 9650
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 8639
    })
    validated: Dataset({
        features: ['audio', 'transcription', 'duration', 'up_votes', 'down_votes', 'age', 'gender', 'accent'],
        num_rows: 46345
    })
})


In [ ]:
datasetdict = datasetdict.rename_column("transcription", "sentence")

In [ ]:
datasetdict = datasetdict.select_columns(["audio", "sentence"])

In [ ]:
from datasets import Audio
datasetdict = datasetdict.cast_column("audio", Audio(sampling_rate=16000))

****AYARRIRIRIRMRM***

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="turkish", task="transcribe")


In [ ]:
datasetdict

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 26501
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9650
    })
})

In [ ]:
del datasetdict["validation"]
del datasetdict["validated"]

In [ ]:
input_str = datasetdict["train"][0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")

Input:                 Kosova başkentinswki yolcu sayısı arttı.
Decoded w/ special:    <|startoftranscript|><|tr|><|transcribe|><|notimestamps|>Kosova başkentinswki yolcu sayısı arttı.<|endoftext|>
Decoded w/out special: Kosova başkentinswki yolcu sayısı arttı.
Are equal:             True


In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="turkish", task="transcribe")


In [ ]:
print(datasetdict["train"][0])

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7aeaf0722f90>, 'sentence': 'Kosova başkentinswki yolcu sayısı arttı.'}


In [ ]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


In [ ]:
datasetdict = datasetdict.map(prepare_dataset, remove_columns=datasetdict.column_names["train"], num_proc=4)


Map (num_proc=4):   0%|          | 0/26501 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/9650 [00:00<?, ? examples/s]

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
model.generation_config.language = "turkish"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None


In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [ ]:
import evaluate

metric = evaluate.load("wer")


In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-hi",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=5000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=datasetdict["train"],
    eval_dataset=datasetdict["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


/tmp/ipython-input-3640337199.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
1000,0.168500,0.254072,20.413862
2000,0.066300,0.253412,19.099957
3000,0.057600,0.239690,18.471645
4000,0.023000,0.248869,17.814693
5000,0.008300,0.243165,17.300945


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3464: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 2

TrainOutput(global_step=5000, training_loss=0.09010562397241592, metrics={'train_runtime': 31535.4344, 'train_samples_per_second': 2.537, 'train_steps_per_second': 0.159, 'total_flos': 8.161471263965184e+19, 'train_loss': 0.09010562397241592, 'epoch': 3.017501508750754})

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

save_dir = "/content/drive/MyDrive/bitirme_projesi/whisper_model"

model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)


[]

In [ ]:
from huggingface_hub import login
login()


In [ ]:
# Önce CLI'de bir kere:
# !huggingface-cli login

repo_name = "whisper-medium-tr-commonvoice-emre"  # kendin bir isim seç

model.push_to_hub(repo_name)
processor.push_to_hub(repo_name)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gh4sl3v/model.safetensors:   0%|          | 99.6kB / 3.06GB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/ietoker/whisper-medium-tr-commonvoice-emre/commit/3ed5e3d21b9a4462e5b0cf8ef120d9485a4230e6', commit_message='Upload processor', commit_description='', oid='3ed5e3d21b9a4462e5b0cf8ef120d9485a4230e6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ietoker/whisper-medium-tr-commonvoice-emre', endpoint='https://huggingface.co', repo_type='model', repo_id='ietoker/whisper-medium-tr-commonvoice-emre'), pr_revision=None, pr_num=None)

In [ ]:
import torch
import torchaudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# 🔁 Modeli ve processor'ü kaydettiğin klasörü buraya yaz
MODEL_DIR = "/content/drive/MyDrive/bitirme_projesi/whisper_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR).to(device)
model.eval()


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias

In [ ]:
def transcribe_audio(audio_path: str) -> str:
    # 1) Ses dosyasını yükle
    waveform, sr = torchaudio.load(audio_path)

    # Stereo ise → mono'ya çevir
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 16 kHz'e resample et (Whisper için şart)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
        sr = 16000

    audio = waveform.squeeze().numpy()

    # 2) Input features + attention mask hazırla
    inputs = processor.feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors="pt",
        return_attention_mask=True,
    )

    input_features = inputs.input_features.to(device)
    attention_mask = inputs.attention_mask.to(device)

    # 3) Model ile çıktı üret
    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
        )

    # 4) Tokenları metne çevir
    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return transcription


In [ ]:
AUDIO_PATH = "/content/drive/MyDrive/bitirme_projesi/Datasets/deneme_2.wav"
text = transcribe_audio(AUDIO_PATH)
print("Transkripsiyon:\n", text)


Transkripsiyon:
 Unutabilirsen sana benden tam beş puan.
